# RigGrade smoke test (free GPU, not a grade)

Checks that the RigGrade GPU code runs on a real card. Costs nothing on a
free Colab or Kaggle GPU. It grades nothing and sends no results anywhere: the
grading endpoint runs inside this notebook with the sale off.

**Before you start:** Runtime > Change runtime type > T4 GPU (Colab), or
Settings > Accelerator > GPU T4 (Kaggle). Then download the repo as a ZIP
from GitHub (Code > Download ZIP) to your computer.

Run the cells top to bottom. Expect 30 to 60 minutes, most of it installing vLLM.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Upload the repo ZIP
On Colab a file picker opens. On Kaggle, add the ZIP as a dataset instead and set `ZIP` to its path.

In [ ]:
import glob, os, zipfile
try:
    from google.colab import files
    uploaded = files.upload()
    ZIP = next(iter(uploaded))
except ImportError:
    ZIP = glob.glob('/kaggle/input/**/riggrade*.zip', recursive=True)[0]
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content' if os.path.isdir('/content') else '/kaggle/working')
ROOT = glob.glob('/content/riggrade*') + glob.glob('/kaggle/working/riggrade*')
ROOT = [p for p in ROOT if os.path.isdir(p)][0]
os.chdir(ROOT)
print('working in', ROOT)

## 3. Install the inference server
This is the slow step (often 5 to 15 minutes). A restart prompt, if shown, can be accepted; then re-run cell 2's `os.chdir` line.

In [ ]:
!pip install -q vllm

## 4. Run the smoke test
About 10 to 20 minutes. Every step is timed and recorded; a failing step does not stop the others.

In [ ]:
!python3 -m runner.smoke --minutes 2

## 4b. Run the free preflight check
About 1 minute. Also writes `preflight-report.json`.

In [ ]:
!python3 -m runner.preflight

## 5. Download the report
Send `smoke-report.json` and `preflight-report.json` back. They hold GPU name, timings and errors only, no keys or account data.

## Record the versions used
vLLM and torch are installed unpinned on a free notebook, so the exact versions are saved with the reports.

In [ ]:
import json, platform, subprocess, sys
def _ver(pkg):
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception as exc:
        return 'not installed (%s)' % type(exc).__name__
versions = {'python': platform.python_version(),
            'vllm': _ver('vllm'), 'torch': _ver('torch'), 'transformers': _ver('transformers'),
            'nvidia_smi': subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version',
                                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()}
open('versions.json', 'w').write(json.dumps(versions, indent=1) + '\n')
print(json.dumps(versions, indent=1))

In [ ]:
import os
missing = [n for n in ('smoke-report.json', 'preflight-report.json', 'versions.json') if not os.path.exists(n)]
assert not missing, 'REPORT MISSING, run not complete: %s' % missing
try:
    from google.colab import files
    files.download('smoke-report.json')
    files.download('preflight-report.json')
    files.download('versions.json')
except ImportError:
    print('Kaggle: find smoke-report.json under Output in the right panel.')